<div align="center">

# Universidad de San Martín

## Infraestructura para Ciencia de Datos

### Licenciatura en Ciencia de Datos

<img src="../logo.jpg" alt="Logo UNSAM" width="300"/>

---

</div>

# 🏗️ Clase 05: La Bóveda (Capa Gold)

---

## 🎯 Objetivo de Hoy

Modelar la **Capa Gold** (la última de la Arquitectura Medallion) usando Star Schema para BI y Wide Tables (ABT) para ML. La Gold es la capa "lista para consumir": optimizada para queries analíticas, dashboards y modelos de Machine Learning.

---


## 📐 1. Modelado Dimensional: El Star Schema

En Silver buscamos integridad/normalización; en **Gold** buscamos **velocidad y facilidad de uso** para análisis de negocio.

### 🌟 ¿Qué es?

Una **Fact Table** en el centro + varias **Dimension Tables** alrededor (como rayos de una estrella). Cada dim se conecta directo al hecho → **menos JOINs**, queries simples y rápidos, ideal para BI (Power BI, Tableau, Looker).

### 📊 Fact Table (hechos)

- Un **evento** del negocio (venta, click, envío). **Larga** (muchas filas) y **estrecha** (pocas columnas).
- **Métricas numéricas** + **FKs** a dimensiones. Inmutable (solo INSERT), crece constantemente.
- Tipos: **transaccional** (1 fila = 1 evento), **periódica/snapshot** (estado a un momento, ej. inventario al cierre), **acumulativa** (proceso multi-etapa).

```
fact_ventas: | venta_id | producto_id | cliente_id | fecha_id | cantidad | monto | margen |
             | 1001     | 42          | 305        | 20240115 | 2        | 50.00 | 20.00  |
```

### 🎯 Dimension Tables (contexto)

- El **contexto** del hecho (quién/qué/dónde/cuándo). **Anchas** (muchas columnas) y **cortas** (pocas filas).
- Atributos textuales/categóricos para **filtrar, agrupar, segmentar**. Cambian lento (SCD).
- La **dim_tiempo** es especial: se pre-calcula con TODOS los días (feriados, trimestres, fin de semana) → acelera queries temporales:

```
dim_tiempo: | fecha_id | fecha      | anio | mes | dia_semana | trimestre | es_feriado |
            | 20240115 | 2024-01-15 | 2024 | 1   | Lunes      | Q1        | FALSE      |
```

`dim_productos`, `dim_clientes`, etc. siguen el mismo patrón: PK simple + atributos descriptivos.

### 🔗 Diagrama

```mermaid
erDiagram
    FACT_VENTAS ||--o{ DIM_PRODUCTOS : producto_id
    FACT_VENTAS ||--o{ DIM_CLIENTES : cliente_id
    FACT_VENTAS ||--o{ DIM_TIEMPO : fecha_id
    FACT_VENTAS {
        int venta_id PK
        int producto_id FK
        int cliente_id FK
        int fecha_id FK
        int cantidad
        float monto
        float margen
    }
    DIM_PRODUCTOS {
        int producto_id PK
        string nombre
        string categoria
        string marca
    }
    DIM_CLIENTES {
        int cliente_id PK
        string nombre
        string ciudad
        string segmento
    }
    DIM_TIEMPO {
        int fecha_id PK
        date fecha
        int anio
        string trimestre
        boolean es_feriado
    }
```

### 📈 Query de negocio típico

```sql
SELECT p.categoria, c.ciudad, SUM(f.monto) AS revenue, AVG(f.monto) AS ticket_prom
FROM fact_ventas f
JOIN dim_productos p ON f.producto_id = p.producto_id
JOIN dim_clientes  c ON f.cliente_id  = c.cliente_id
JOIN dim_tiempo    t ON f.fecha_id    = t.fecha_id
WHERE t.trimestre = 'Q1' AND t.anio = 2024
GROUP BY p.categoria, c.ciudad
ORDER BY revenue DESC;
```
✅ Pocos JOINs: simple y rápido para analistas.

### 🆚 Star vs Snowflake

| | Star Schema | Snowflake |
|:---|:---|:---|
| **Normalización** | Dims desnormalizadas | Dims normalizadas |
| **Complejidad / JOINs** | Simple, menos JOINs | Más complejo |
| **Performance** | Más rápido | Más lento |
| **Espacio** | Más redundancia | Menos |
| **Uso** | BI / Reporting | DWH legacy |

**Recomendación:** en Gold, **Star Schema**. El espacio es barato; la simplicidad es cara.

---

## 🔍 1.2 Fact vs Dimension: cómo decidir

### 📊 Comparación

| Aspecto | Fact | Dimension |
|:---|:---|:---|
| **Contenido** | Eventos/transacciones | Contexto descriptivo |
| **Columnas** | Métricas numéricas + FKs | Atributos textuales/categóricos |
| **Forma** | Larga y estrecha | Corta y ancha |
| **Tamaño** | Crece sin parar (M/B filas) | Estable (miles) |
| **Operaciones** | Solo INSERT (inmutable) | INSERT + UPDATE (SCD) |
| **Keys** | PK compuesta/surrogate, muchas FKs | PK simple, sin FKs |
| **Ejemplo** | Una venta de $150 el 15/01 | El producto "Mouse Gamer, Logitech" |

**Regla rápida:** ¿es un **número que se suma** o una **medida del evento** (monto, cantidad, score)? → **FACT**. ¿Es **descriptivo/categórico** y se usa para **filtrar o agrupar** (nombre, categoría, ciudad)? → **DIMENSION**.

**Degenerate dimensions** (atributo no numérico que no justifica una dim): `numero_factura` → FACT (único por transacción); `tipo_pago` → FACT (pocos valores). Texto libre (`comentarios`) → ni fact ni dim (va a un data lake).

### 📏 Granularidad

> [!IMPORTANT]
> **Regla de oro:** definí qué representa **UNA fila** de la fact ANTES de crearla. Cambiarla después es carísimo.

Ej.: `fact_ventas` a grano "línea de venta" (1 fila = 1 producto en 1 ticket) vs grano "transacción" (1 fila = 1 ticket completo).

### 🔢 Tipos de métricas

- **Additive** — se suman en *todas* las dimensiones (`monto`, `cantidad`).
- **Semi-additive** — se suman en algunas pero **NO en tiempo** (`saldo_cuenta`, `inventario`: sumás por producto, no por fecha → usá promedio o último valor).
- **Non-additive** — nunca se suman, solo ratios/promedios (`temperatura`, `tasa_conversion`).

---

## 🏢 2. La Capa Semántica (Semantic Layer)

Modelar el **Star Schema** es el primer paso, pero en arquitecturas modernas de datos, el objetivo final es la **Capa Semántica**. Es donde transformamos los datos físicos en conceptos de negocio.

### 🧠 ¿Qué es una Capa Semántica?

Es una abstracción que se sitúa por encima del Star Schema y define las **métricas** y **dimensiones** de forma única para toda la organización. 

**El problema:** Si un analista de marketing calcula el *Revenue* sumando `monto` y otro analista de finanzas lo calcula restando `descuentos`, tenemos dos versiones de la verdad. 

**La solución:** Definir la métrica *Revenue* una sola vez en la Capa Semántica.

| Atributo | Modelado Físico (Star Schema) | Capa Semántica (Semantic Layer) |
|:---|:---|:---|
| **Donde vive** | Tablas de SQL (`fact_ventas`) | Código (dbt Semantic Layer, LookML, MetricStore) |
| **Estructura** | Tablas y Columnas | Métricas y Dimensiones |
| **Fórmula** | Repetida en cada query SQL | Definida una sola vez centralizada |
| **JOINs** | Manuales en cada query | Automáticos según el contexto |

### 🎯 Niveles de Abstracción en Gold

1. **Capa de Modelo (Physical)**: Tablas `fact` y `dim` optimizadas para almacenamiento y JOINs.
2. **Capa Semántica (Logical)**: Definiciones de métricas (ej. `total_revenue = sum(amount)`) y jerarquías (ej. Pais -> Provincia -> Ciudad).
3. **Capa de Presentación (BI)**: Dashboards consumiendo las métricas de la capa lógica.

> [!IMPORTANT]
> **Medallion Gold Standard:** Una buena capa Gold no solo tiene tablas limpias, tiene **métricas gobernadas**. Si cambias la definición de *Margen Bruto*, lo haces en un solo lugar y todos los reportes se actualizan automáticamente.

---

## 🤖 3. ML Gold: la Wide Table (ABT)

Los modelos de ML no quieren el Star Schema normalizado: necesitan una **Wide Table** (ABT — *Analytical Base Table*): **1 fila = 1 entidad** (cliente/producto), **1 columna = 1 feature**, **sin JOINs**, lista para `scikit-learn`/`TensorFlow`. Es "aplanar" el modelo dimensional vía **feature engineering**.

### 📊 Ejemplo: ABT de Churn

Predecir si un cliente deja de comprar en 30 días. Features = demográficos (de `dim_clientes`) + comportamiento agregado (de `fact_ventas`: total compras, gasto, frecuencia) + recencia (días desde última compra) + **target** (churn sí/no).

```sql
CREATE TABLE gold_abt_churn AS
WITH agg AS (
  SELECT cliente_id,
         COUNT(venta_id) AS total_compras,
         SUM(monto)      AS gasto_total,
         AVG(monto)      AS ticket_promedio,
         MAX(fecha)      AS ultima_compra
  FROM fact_ventas GROUP BY cliente_id
)
SELECT c.cliente_id, c.edad, c.ciudad, c.segmento,
       COALESCE(a.total_compras, 0)   AS total_compras,
       COALESCE(a.gasto_total, 0)     AS gasto_total,
       COALESCE(a.ticket_promedio, 0) AS ticket_promedio,
       COALESCE(DATE_PART('day', CURRENT_DATE - a.ultima_compra), 999) AS dias_sin_comprar,
       CASE WHEN a.ultima_compra IS NULL
             OR DATE_PART('day', CURRENT_DATE - a.ultima_compra) > 60
            THEN 1 ELSE 0 END AS churn   -- target
FROM dim_clientes c
LEFT JOIN agg a ON c.cliente_id = a.cliente_id;
```

```
| cliente_id | edad | segmento | total_compras | gasto_total | dias_sin_comprar | churn |
| 101        | 32   | Premium  | 45            | 15000.50    | 5                | 0     |
| 103        | 35   | Premium  | 0             | 0.00        | 999              | 1     |
```

### 🧪 Uso en Python

```python
df = pd.read_sql("SELECT * FROM gold_abt_churn", engine)
X = pd.get_dummies(df.drop(columns=["cliente_id", "churn"]))
y = df["churn"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_tr, y_tr)
print(f"Accuracy: {model.score(X_te, y_te):.2%}")
```

### 🏗️ ABT vs Star Schema

| | Star Schema (BI) | Wide Table / ABT (ML) |
|:---|:---|:---|
| **Propósito** | Dashboards/reportes | Entrenar modelos |
| **Estructura** | Normalizada (fact+dims) | Desnormalizada (flat) |
| **Operación** | Queries con JOINs | Sin JOINs, directo al modelo |
| **Actualización** | Diaria/tiempo real | Batch (semanal/mensual) |

**Feature engineering típico:** agregaciones (`SUM`/`AVG`/`COUNT`/`STDDEV`), temporales (recency/frequency, ventanas 30/60/90 d), ratios (tasa de descuento/devolución), one-hot de categóricas.

> ⚠️ Cuidado con **data leakage** (no mezclar features con el target), features con >90% nulos e *imbalance* del target. Se profundiza en **clase 06**.

---

### 🧭 Nota arquitectural: ¿por qué Gold NO usa Pydantic dinámico?

En clase04 (Silver) cargamos el contrato YAML con `build_pydantic_from_contract(load_contract("crypto_markets.yaml"))` y validamos **cada fila** contra ese modelo — separando lo válido de la quarantine.

En Gold **no aplicamos esa validación**, y es una decisión consciente. La unidad de validación cambia:

| Capa | Unidad validada | Herramienta | Qué falla → |
|---|---|---|---|
| **Silver** | Fila individual (`{symbol: "BTC", price: 50000, ...}`) | Pydantic dinámico | Quarantine con `quarantine_reason` |
| **Gold** | Schema completo (que la tabla tenga las columnas esperadas, los joins no rompan, no haya huérfanos) | Assertions + LEFT JOIN | El DAG falla — no hay quarantine fila a fila |

> 🎯 Para Gold lo que importa es **integridad referencial** (que los joins no dejen huérfanos) — no si una fila tiene `current_price = NaN`. Eso ya lo filtró Silver.
>
> En clase06 (workshop) vas a ver cómo **dbt** formaliza este concepto: los tests dbt (`unique`, `not_null`, `relationships`) son la versión SQL declarativa de las assertions que acá escribimos en pandas.

---

## 🎓 4. Best Practices — Capa Gold

### ✅ DO

| Práctica | Razón |
|:---|:---|
| **dim_tiempo completa** (todos los días, feriados, trimestres) | Acelera queries de tendencias |
| **Surrogate keys** en dims SCD | Simplifica JOINs con historia |
| **Desnormalizar** el Star Schema | Queries simples para analistas |
| **Particionar fact tables por fecha** | Performance en tablas grandes |
| **Metadatos de carga** (`_processed_at`, `_source_table`) | Auditoría / linaje |
| **Naming claro** (`fact_*`, `dim_*`, `*_abt`) | Comprensión |
| **Pre-calcular KPIs en Gold** (no en la BI tool) | Consistencia entre reportes |

### ❌ DON'T

| Anti-práctica | Problema | Alternativa |
|:---|:---|:---|
| Normalizar dims (Snowflake) | Queries lentos/complejos | Star desnormalizado |
| Dims sin SCD | Reportes históricos mal | SCD Tipo 2 si la dim cambia |
| Fact sin granularidad clara | Confusión en análisis | Definir "¿qué es 1 fila?" |
| Mezclar BI y ML en una tabla | Tabla inmanejable | Star para BI, ABT para ML |
| Calcular métricas en la BI tool | Inconsistencias | Pre-calcular en Gold |
| No particionar tablas grandes | Queries lentos | Partición por fecha/región |

### 🏗️ BI vs ML desde Silver

```
Silver (normalizado)
   ├─→ Gold BI  (Star Schema):  fact_ventas + dim_clientes/productos/tiempo
   └─→ Gold ML  (Wide Tables):  gold_abt_churn, gold_abt_productos
```

### ⚡ Performance (cuando la fact crece)

| Técnica | Cuándo | Impacto |
|:---|:---|:---|
| Índices en FK | JOINs frecuentes | 10-50x |
| Particionamiento | fact > 10M filas | 10-100x |
| Materialized Views | agregaciones costosas repetidas | ~100x |
| `VACUUM` / `ANALYZE` | tras cargas grandes | 2-5x |

> Una **materialized view** pre-calcula un KPI caro (ej. revenue mensual por categoría) y se refresca en el scheduler → queries instantáneos. Particionar por rango de fecha habilita *partition pruning* (el query solo lee la partición relevante).

### 🔍 Calidad en Gold (qué testear)

- **Integridad referencial**: 0 filas de fact sin su dim (`LEFT JOIN ... WHERE dim.pk IS NULL`).
- **Sin duplicados** en la PK de la fact.
- **Business key** de las dims sin nulos.
- **SCD**: a lo sumo 1 versión `es_actual = TRUE` por entidad.

Esto se formaliza con **dbt tests** (`unique`, `not_null`, `relationships`) en **clase 06**.

---

## 🐳 5. Pipeline Gold con Airflow — DAGs Pedagógicos

Hasta acá vimos los conceptos: Star Schema, dimensiones, fact tables, ABT, feature engineering. Ahora **bajamos esos conceptos a DAGs reales** sobre datos sintéticos.

| # | DAG generado | Path destino | Qué introduce |
|---|---|---|---|
| 01 | `gold_01_star_basico.py` | `stack/dags/03-gold/` | Star Schema: `dim_producto_demo` + `dim_tiempo_demo` + `fact_ventas_demo` con FKs (surrogate keys, dimensiones, fact con FKs) |
| 02 | `gold_02_abt.py` | `stack/dags/03-gold/` | ABT (1 fila por cliente): features derivadas + segmentación con `pd.cut` + verificación de integridad |

Después de ejecutar las celdas, los DAGs aparecen en Airflow UI (`localhost:8080`). Filtrá por tag **`gold`** para verlos juntos.

> **Convención de naming**: `gold_NN_xxx.py` con prefijo letra (igual que `bronze_NN_xxx.py` en clase03 y `silver_NN_xxx.py` en clase04). Evita el bug histórico de Airflow con archivos que arrancan con dígito.
>
> **Convención de tags**: sintéticos didácticos llevan `tags=["gold"]`. El DAG productivo (crypto) lleva `tags=["prod", "gold", "crypto"]` — filtrá por `prod` en la UI para ver sólo los productivos.

---

### 🛠️ Runbook — DAG 1: `gold_01_star_basico.py`

🎯 **Qué hace**: toma `silver.ventas_demo` (datos limpios sintéticos generados por la primera task), construye 2 dimensiones (`dim_producto_demo`, `dim_tiempo_demo`) y la `fact_ventas_demo` con las foreign keys correspondientes. Es el patrón clásico de Star Schema para BI.

📋 **Pasos para correrlo**:
1. Ejecutar la siguiente celda (`%%writefile`) → escribe el DAG en `stack/dags/03-gold/gold_01_star_basico.py`
2. Esperar 10-30s — Airflow detecta el archivo automáticamente (volumen montado)
3. En Airflow UI (`localhost:8080`), filtrá por tag `gold` y activá el toggle del DAG `gold_01_star_basico`
4. Trigger manual (botón "play")

👀 **Qué observar específicamente**:
- En Postgres: las 3 tablas `gold.dim_producto_demo`, `gold.dim_tiempo_demo`, `gold.fact_ventas_demo` se crean con datos
- `dim_producto_demo.producto_id` es una **surrogate key** (1, 2, 3...) — NO viene del source, se genera en el DAG con `range(1, len(df) + 1)`
- `fact_ventas_demo` tiene `producto_id` y `fecha_id` como FKs apuntando a las dims, más `monto_total` derivado (`precio × cantidad`)
- En el grafo del DAG: `prepare_silver → [build_dim_producto, build_dim_tiempo] → build_fact_ventas` (las dims corren en paralelo, la fact espera a ambas)

In [ ]:
%%writefile ../stack/dags/03-gold/gold_01_star_basico.py

"""
DAG pedagogico: Gold Star Schema Basico
Clase 05 - Construccion de un star schema dimensional.

Pipeline didactico:
  silver.ventas_demo -> dim_producto_demo + dim_tiempo_demo + fact_ventas_demo

Conceptos clave:
  - Surrogate keys (producto_id autogenerado)
  - Dimension tables (1 fila = 1 entidad descriptiva)
  - Fact table (1 fila = 1 evento, con FKs a dimensiones)
"""

from airflow.decorators import dag, task
from datetime import datetime
import os


DB_URI = (
    f"postgresql+psycopg2://"
    f"{os.getenv('SOURCE_DB_USER', 'admin')}:"
    f"{os.getenv('SOURCE_DB_PASS', 'admin')}@"
    f"{os.getenv('SOURCE_DB_HOST', 'data_warehouse')}:5432/"
    f"{os.getenv('SOURCE_DB_NAME', 'InfraCienciaDatos')}"
)


@dag(
    dag_id="gold_01_star_basico",
    start_date=datetime(2024, 1, 1),
    schedule=None,
    catchup=False,
    tags=["gold"],
    doc_md="DAG didactico: Star Schema (dim + fact) sobre datos sinteticos.",
)
def gold_01_star_basico():

    @task
    def prepare_silver():
        """Crea silver.ventas_demo con datos limpios (10 filas, 5 clientes)."""
        import pandas as pd
        import sqlalchemy

        data = {
            "venta_id": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
            "cliente_id": [101, 102, 103, 101, 104, 105, 102, 103, 101, 104],
            "producto": ["Laptop", "Mouse", "Teclado", "Monitor", "Auriculares",
                         "Laptop", "Mouse", "Teclado", "Auriculares", "Monitor"],
            "precio": [1500.50, 25.00, 75.00, 350.00, 120.00,
                       1450.00, 30.00, 80.00, 110.00, 360.00],
            "cantidad": [1, 2, 1, 1, 3, 1, 4, 1, 2, 1],
            "fecha": ["2024-01-15", "2024-01-16", "2024-01-17", "2024-01-18", "2024-01-19",
                      "2024-02-01", "2024-02-05", "2024-02-10", "2024-03-01", "2024-03-05"],
        }
        df = pd.DataFrame(data)
        df["fecha"] = pd.to_datetime(df["fecha"])

        engine = sqlalchemy.create_engine(DB_URI)
        with engine.begin() as conn:
            conn.execute(sqlalchemy.text("CREATE SCHEMA IF NOT EXISTS silver;"))
        df.to_sql("ventas_demo", engine, schema="silver", if_exists="replace", index=False)
        print(f"silver.ventas_demo: {len(df)} filas listas.")

    @task
    def build_dim_producto():
        """Productos unicos -> gold.dim_producto_demo (con producto_id surrogate)."""
        import pandas as pd
        import sqlalchemy

        engine = sqlalchemy.create_engine(DB_URI)
        df = pd.read_sql(
            "SELECT DISTINCT producto FROM silver.ventas_demo ORDER BY producto",
            engine,
        )
        df["producto_id"] = range(1, len(df) + 1)
        df = df[["producto_id", "producto"]]

        with engine.begin() as conn:
            conn.execute(sqlalchemy.text("CREATE SCHEMA IF NOT EXISTS gold;"))
        df.to_sql("dim_producto_demo", engine, schema="gold", if_exists="replace", index=False)
        print(f"gold.dim_producto_demo: {len(df)} productos.")

    @task
    def build_dim_tiempo():
        """Fechas unicas + atributos temporales -> gold.dim_tiempo_demo."""
        import pandas as pd
        import sqlalchemy

        engine = sqlalchemy.create_engine(DB_URI)
        df = pd.read_sql(
            "SELECT DISTINCT fecha FROM silver.ventas_demo ORDER BY fecha",
            engine,
        )
        df["fecha"] = pd.to_datetime(df["fecha"])
        df["fecha_id"] = df["fecha"].dt.strftime("%Y%m%d").astype(int)
        df["anio"] = df["fecha"].dt.year
        df["mes"] = df["fecha"].dt.month
        df["trimestre"] = df["fecha"].dt.quarter
        df["dia_semana"] = df["fecha"].dt.day_name()
        df = df[["fecha_id", "fecha", "anio", "mes", "trimestre", "dia_semana"]]

        df.to_sql("dim_tiempo_demo", engine, schema="gold", if_exists="replace", index=False)
        print(f"gold.dim_tiempo_demo: {len(df)} fechas.")

    @task
    def build_fact_ventas():
        """Hechos con FKs a dim_producto_demo y dim_tiempo_demo + monto_total derivado."""
        import pandas as pd
        import sqlalchemy

        engine = sqlalchemy.create_engine(DB_URI)
        df = pd.read_sql("SELECT * FROM silver.ventas_demo", engine)
        dim_p = pd.read_sql("SELECT * FROM gold.dim_producto_demo", engine)

        df["fecha"] = pd.to_datetime(df["fecha"])
        df["fecha_id"] = df["fecha"].dt.strftime("%Y%m%d").astype(int)
        df = df.merge(dim_p, on="producto", how="left")
        df["monto_total"] = df["precio"] * df["cantidad"]

        fact = df[["venta_id", "fecha_id", "producto_id", "cliente_id",
                   "precio", "cantidad", "monto_total"]]
        fact.to_sql("fact_ventas_demo", engine, schema="gold", if_exists="replace", index=False)
        print(f"gold.fact_ventas_demo: {len(fact)} filas.")

    silver_done = prepare_silver()
    p = build_dim_producto()
    t = build_dim_tiempo()
    silver_done >> [p, t]
    [p, t] >> build_fact_ventas()


gold_01_star_basico()


### 🛠️ Runbook — DAG 2: `gold_02_abt.py`

🎯 **Qué hace**: agrupa `silver.ventas_demo` por `cliente_id` y deriva una **ABT (wide table) lista para ML** — 1 fila por cliente, N columnas de features. Cierra con una task `verify_integrity` que detecta clientes huérfanos vía LEFT JOIN.

📋 **Pasos para correrlo**:
1. Ejecutar la siguiente celda (`%%writefile`) → escribe el DAG en `stack/dags/03-gold/gold_02_abt.py`
2. Esperar 10-30s para que Airflow lo detecte
3. En Airflow UI, filtrá por tag `gold` y activá `gold_02_abt`
4. Trigger manual

👀 **Qué observar específicamente**:
- En Postgres: `gold.abt_clientes_demo` con **una fila por `cliente_id` único** (no por venta) — el `groupby` colapsa transacciones múltiples del mismo cliente
- Las features derivadas: `total_compras` (count), `monto_total` (sum), `ticket_promedio` (mean redondeado a 2 decimales), `recencia_dias` (días desde la última compra hasta hoy)
- `segmento_valor` categórica creada con `pd.cut(monto_total, bins=[0, 100, 1000, inf])` → cada cliente queda etiquetado como `Bronze`, `Silver` o `Gold`
- Logs de `verify_integrity`: si hay clientes huérfanos imprime un WARNING — si todo está bien imprime `"OK: integridad referencial verificada."`

In [ ]:
%%writefile ../stack/dags/03-gold/gold_02_abt.py

"""
DAG pedagogico: Gold ABT (Wide Table) para ML
Clase 05 - Feature Engineering: 1 fila por entidad con metricas agregadas.

Pipeline didactico:
  silver.ventas_demo -> gold.abt_clientes_demo (1 fila por cliente)

Conceptos clave:
  - ABT (Analytical Base Table): tabla ancha, denormalizada
  - Feature engineering: derivar features agregados (count, sum, mean, time-based)
  - Categorizacion con pd.cut (segmentacion por valor)
  - Verificacion de integridad referencial
"""

from airflow.decorators import dag, task
from datetime import datetime
import os


DB_URI = (
    f"postgresql+psycopg2://"
    f"{os.getenv('SOURCE_DB_USER', 'admin')}:"
    f"{os.getenv('SOURCE_DB_PASS', 'admin')}@"
    f"{os.getenv('SOURCE_DB_HOST', 'data_warehouse')}:5432/"
    f"{os.getenv('SOURCE_DB_NAME', 'InfraCienciaDatos')}"
)


@dag(
    dag_id="gold_02_abt",
    start_date=datetime(2024, 1, 1),
    schedule=None,
    catchup=False,
    tags=["gold"],
    doc_md="DAG didactico: ABT (wide table) con feature engineering para ML.",
)
def gold_02_abt():

    @task
    def prepare_silver():
        """Crea silver.ventas_demo (mismo set que gold_01_star_basico)."""
        import pandas as pd
        import sqlalchemy

        data = {
            "venta_id": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
            "cliente_id": [101, 102, 103, 101, 104, 105, 102, 103, 101, 104],
            "producto": ["Laptop", "Mouse", "Teclado", "Monitor", "Auriculares",
                         "Laptop", "Mouse", "Teclado", "Auriculares", "Monitor"],
            "precio": [1500.50, 25.00, 75.00, 350.00, 120.00,
                       1450.00, 30.00, 80.00, 110.00, 360.00],
            "cantidad": [1, 2, 1, 1, 3, 1, 4, 1, 2, 1],
            "fecha": ["2024-01-15", "2024-01-16", "2024-01-17", "2024-01-18", "2024-01-19",
                      "2024-02-01", "2024-02-05", "2024-02-10", "2024-03-01", "2024-03-05"],
        }
        df = pd.DataFrame(data)
        df["fecha"] = pd.to_datetime(df["fecha"])

        engine = sqlalchemy.create_engine(DB_URI)
        with engine.begin() as conn:
            conn.execute(sqlalchemy.text("CREATE SCHEMA IF NOT EXISTS silver;"))
        df.to_sql("ventas_demo", engine, schema="silver", if_exists="replace", index=False)

    @task
    def build_abt_clientes():
        """Agrupa por cliente_id y deriva features de comportamiento."""
        import pandas as pd
        import sqlalchemy

        engine = sqlalchemy.create_engine(DB_URI)
        df = pd.read_sql("SELECT * FROM silver.ventas_demo", engine)
        df["fecha"] = pd.to_datetime(df["fecha"])
        df["monto"] = df["precio"] * df["cantidad"]

        ahora = pd.Timestamp.now().normalize()

        abt = df.groupby("cliente_id").agg(
            total_compras=("venta_id", "count"),
            monto_total=("monto", "sum"),
            ticket_promedio=("monto", "mean"),
            ultima_compra=("fecha", "max"),
            primera_compra=("fecha", "min"),
        ).reset_index()

        abt["recencia_dias"] = (ahora - abt["ultima_compra"]).dt.days
        abt["ticket_promedio"] = abt["ticket_promedio"].round(2)

        # Segmentacion por monto_total (Bronze / Silver / Gold)
        abt["segmento_valor"] = pd.cut(
            abt["monto_total"],
            bins=[0, 100, 1000, float("inf")],
            labels=["Bronze", "Silver", "Gold"],
        ).astype(str)

        with engine.begin() as conn:
            conn.execute(sqlalchemy.text("CREATE SCHEMA IF NOT EXISTS gold;"))
        abt.to_sql("abt_clientes_demo", engine, schema="gold", if_exists="replace", index=False)
        print(f"gold.abt_clientes_demo: {len(abt)} clientes con features.")

    @task
    def verify_integrity():
        """LEFT JOIN para detectar clientes huerfanos."""
        import pandas as pd
        import sqlalchemy

        engine = sqlalchemy.create_engine(DB_URI)
        q = """
            SELECT COUNT(DISTINCT v.cliente_id) AS huerfanos
            FROM silver.ventas_demo v
            LEFT JOIN gold.abt_clientes_demo a ON v.cliente_id = a.cliente_id
            WHERE a.cliente_id IS NULL
        """
        n = int(pd.read_sql(q, engine).iloc[0, 0])
        if n > 0:
            print(f"WARNING: {n} clientes huerfanos detectados.")
        else:
            print("OK: integridad referencial verificada.")

    silver_done = prepare_silver()
    abt_done = build_abt_clientes()
    silver_done >> abt_done >> verify_integrity()


gold_02_abt()


## 📊 6. Visualización — Dashboard sobre Gold

Los datos en Gold sirven a dos audiencias: **BI** (Star Schema, queries SQL) y **ML** (ABT, features). Esta sección genera una **página de Streamlit** que muestra ambos.

### 🛠️ Runbook — Página dashboard: `5_Gold_Demo_Ventas.py`

🎯 **Qué hace**: escribe una nueva página Streamlit en `stack/dashboard/pages/5_Gold_Demo_Ventas.py` que lee las 4 tablas Gold sintéticas y las renderiza con métricas + 3 gráficos (ventas por producto, ventas por mes, segmentación ABT).

📋 **Pasos para correrlo**:
1. **Pre-requisito**: tener corridos antes los DAGs `gold_01_star_basico` y `gold_02_abt` (las tablas `gold.dim_*`, `gold.fact_*`, `gold.abt_clientes_demo` deben existir). Si faltan, la página renderiza un `st.warning` y se corta.
2. Ejecutar la siguiente celda (`%%writefile`) → escribe la página en `stack/dashboard/pages/5_Gold_Demo_Ventas.py`
3. Refrescar `localhost:8501` (Streamlit detecta automáticamente archivos `.py` nuevos en `pages/`)
4. En el menú lateral, aparece una nueva entrada **"Demo Pedagogico - Star Schema y ABT"** — clickeala

> **Nota**: el prefijo `7_` al inicio del nombre **NO es un bug**: es la **convención de orden de Streamlit** (los números ordenan las páginas en el sidebar). Distinto de Airflow donde el dígito al inicio causa skipping silencioso.

👀 **Qué observar específicamente**:
- **4 métricas en la barra superior**: Productos, Fechas, Ventas, Clientes — cada una es un `COUNT(*)` de su tabla
- **Gráfico de barras**: monto total vendido por producto (join `fact_ventas_demo` + `dim_producto_demo`)
- **Gráfico de líneas**: monto vendido por mes (join `fact_ventas_demo` + `dim_tiempo_demo`)
- **Donut chart + tabla**: segmentación de clientes por `segmento_valor` (Bronze / Silver / Gold) leyendo `gold.abt_clientes_demo`
- Si refrescás y NO aparece la página, chequeá los logs de Streamlit con `docker compose logs streamlit | tail -20`

In [ ]:
%%writefile ../stack/dashboard/pages/5_Gold_Demo_Ventas.py

"""
Pagina demo pedagogica - Star Schema (BI) y ABT (ML) generados por los
DAGs didacticos de clase05 (gold_01_star_basico, gold_02_abt).

Conecta con las tablas (sufijo _demo, no colisionan con crypto_gold):
  - gold.dim_producto_demo, gold.dim_tiempo_demo, gold.fact_ventas_demo  (Star Schema / BI)
  - gold.abt_clientes_demo                                               (ABT / ML)
"""

import streamlit as st
import plotly.express as px
import pandas as pd
from db import load_table, get_row_count, table_exists, get_engine

st.header("Demo Pedagogico - Star Schema y ABT")
st.caption("Datos sinteticos de `gold_01_star_basico` y `gold_02_abt`. "
           "Las dos audiencias de Gold: **BI** (Star Schema) y **ML** (ABT).")

engine = get_engine()
needed = [("gold", "fact_ventas_demo"), ("gold", "dim_producto_demo"),
          ("gold", "dim_tiempo_demo"), ("gold", "abt_clientes_demo")]
missing = [f"{s}.{t}" for s, t in needed if not table_exists(engine, s, t)]
if missing:
    st.warning(
        "Faltan tablas: " + ", ".join(missing) +
        ". Ejecuta los DAGs `gold_01_star_basico` y `gold_02_abt` desde Airflow."
    )
    st.stop()

col1, col2, col3, col4 = st.columns(4)
col1.metric("Productos", get_row_count("gold", "dim_producto_demo"))
col2.metric("Fechas", get_row_count("gold", "dim_tiempo_demo"))
col3.metric("Ventas", get_row_count("gold", "fact_ventas_demo"))
col4.metric("Clientes", get_row_count("gold", "abt_clientes_demo"))
st.divider()

tab_bi, tab_ml = st.tabs(["\U0001F4CA BI — Star Schema", "\U0001F916 ML — ABT"])

# =============================================================
# TAB BI: Star Schema = JOIN fact + dim + agregacion
# =============================================================
with tab_bi:
    st.caption("Consumo BI: se hace **JOIN** fact + dim y se **agrega**. "
               "Es lo que hace un dashboard / reporte.")

    top_n = st.slider("Top N productos por monto", 3, 12, 8)
    q_vp = """
        SELECT p.producto, SUM(f.monto_total) AS total
        FROM gold.fact_ventas_demo f
        JOIN gold.dim_producto_demo p ON f.producto_id = p.producto_id
        GROUP BY p.producto
        ORDER BY total DESC
    """
    df_vp = pd.read_sql(q_vp, engine).head(top_n)
    if not df_vp.empty:
        fig = px.bar(df_vp, x="producto", y="total", color="total",
                     color_continuous_scale="Blues",
                     labels={"total": "Monto total ($)", "producto": ""},
                     title=f"Top {top_n} productos por monto")
        fig.update_layout(height=350, showlegend=False)
        st.plotly_chart(fig, use_container_width=True)

    q_vm = """
        SELECT t.anio, t.mes, SUM(f.monto_total) AS monto
        FROM gold.fact_ventas_demo f
        JOIN gold.dim_tiempo_demo t ON f.fecha_id = t.fecha_id
        GROUP BY t.anio, t.mes
        ORDER BY t.anio, t.mes
    """
    df_vm = pd.read_sql(q_vm, engine)
    if not df_vm.empty:
        df_vm["periodo"] = (df_vm["anio"].astype(str) + "-"
                            + df_vm["mes"].astype(str).str.zfill(2))
        fig2 = px.line(df_vm, x="periodo", y="monto", markers=True,
                       labels={"monto": "Monto total ($)", "periodo": ""},
                       title="Ventas por mes (JOIN con dim_tiempo_demo)")
        fig2.update_layout(height=300)
        st.plotly_chart(fig2, use_container_width=True)

# =============================================================
# TAB ML: ABT = 1 fila por cliente, features ya derivadas, SIN JOINs
# =============================================================
with tab_ml:
    st.caption("Consumo ML: **1 fila = 1 cliente**, features ya derivadas, "
               "**sin JOINs** (lista para entrenar un modelo).")

    abt = load_table("gold", "abt_clientes_demo")
    if not abt.empty:
        colors = {"Bronze": "#cd7f32", "Silver": "#c0c0c0", "Gold": "#ffd700"}
        c1, c2 = st.columns(2)
        with c1:
            seg = abt["segmento_valor"].value_counts().reset_index()
            seg.columns = ["segmento", "clientes"]
            fig_seg = px.pie(seg, names="segmento", values="clientes", hole=0.4,
                             color="segmento", color_discrete_map=colors,
                             title="Clientes por segmento")
            fig_seg.update_layout(height=300, margin=dict(t=40, b=10))
            st.plotly_chart(fig_seg, use_container_width=True)
        with c2:
            ins = (abt.groupby("segmento_valor", as_index=False)["ticket_promedio"]
                      .mean().sort_values("ticket_promedio", ascending=False))
            fig_ins = px.bar(ins, x="segmento_valor", y="ticket_promedio",
                             color="segmento_valor", color_discrete_map=colors,
                             labels={"ticket_promedio": "Ticket promedio ($)",
                                     "segmento_valor": ""},
                             title="Ticket promedio por segmento")
            fig_ins.update_layout(height=300, showlegend=False,
                                  margin=dict(t=40, b=10))
            st.plotly_chart(fig_ins, use_container_width=True)

        segs = ["Todos"] + sorted(abt["segmento_valor"].dropna().unique().tolist())
        sel = st.selectbox("Filtrar tabla por segmento", segs)
        view = abt if sel == "Todos" else abt[abt["segmento_valor"] == sel]
        st.dataframe(view.sort_values("monto_total", ascending=False),
                     use_container_width=True, hide_index=True)

st.divider()
st.caption(
    "**Patron didactico:** estos DAGs generan tablas sinteticas hardcoded. "
    "El DAG productivo `crypto_gold` aplica el MISMO patron (Star Schema + ABT) "
    "sobre datos reales de criptomonedas, y las paginas Gold reales del "
    "dashboard lo consumen igual que esta demo."
)

### 🛠️ Runbook — 4 páginas Gold productivas del dashboard

Estas celdas `%%writefile` regeneran las **4 páginas Gold del dashboard** en `stack/dashboard/pages/` (numeradas `1_`–`4_`). Leen las tablas **productivas** que crea `crypto_gold` (`gold.dim_crypto`, `gold.fact_crypto_markets`, `gold.fact_global_market`, `gold.gold_abt_crypto`) — datos reales de cripto, **no** las `*_demo`.

🎯 **Por qué desde el notebook**: quedan documentadas y reproducibles acá (igual que la página demo `5_Gold_Demo_Ventas.py` y los DAGs `gold_0*`). Para verlas: ejecutá estas celdas y refrescá Streamlit (`localhost:8501`).

📋 **Pre-requisito**: tener `crypto_gold` corrido (tablas `gold.*` pobladas).

In [ ]:
%%writefile ../stack/dashboard/pages/1_Gold_Resumen_Mercado.py

"""
Resumen del Mercado - Vista ejecutiva con KPIs y Top 10.
Objetivo: responder "como esta el mercado crypto hoy?"
"""

import streamlit as st
import plotly.express as px
from db import load_table, show_last_updated_badge

st.header("Gold - Resumen del Mercado")
show_last_updated_badge("gold", "fact_crypto_markets", col="_loaded_at")

# --- Cargar datos ---
fact = load_table("gold", "fact_crypto_markets")
dim = load_table("gold", "dim_crypto")
fact_global = load_table("gold", "fact_global_market")

if fact.empty:
    st.warning("No hay datos en gold todavia. Ejecuta el pipeline desde Airflow.")
    st.stop()

df = fact.copy()
if not dim.empty and "crypto_id" in dim.columns:
    df = df.merge(dim[["crypto_id", "name", "symbol"]], on="crypto_id", how="left")

# =============================================================
# KPIs GLOBALES
# =============================================================
col1, col2, col3, col4 = st.columns(4)

total_market_cap = df["market_cap"].sum() if "market_cap" in df.columns else 0
total_volume = df["total_volume"].sum() if "total_volume" in df.columns else 0
n_cryptos = df["crypto_id"].nunique() if "crypto_id" in df.columns else 0

col1.metric("Market Cap Total", f"${total_market_cap / 1e9:,.1f}B")
col2.metric("Volumen 24h", f"${total_volume / 1e9:,.1f}B")
col3.metric("Criptomonedas", n_cryptos)

if not fact_global.empty and "btc_dominance" in fact_global.columns:
    col4.metric("BTC Dominancia", f"{fact_global['btc_dominance'].iloc[-1]:.1f}%")
else:
    avg_change = df["price_change_percentage_24h"].mean() if "price_change_percentage_24h" in df.columns else 0
    col4.metric("Cambio Prom. 24h", f"{avg_change:+.2f}%")

# --- Global extra ---
if not fact_global.empty:
    gcol1, gcol2, gcol3 = st.columns(3)
    if "total_market_cap_usd" in fact_global.columns:
        gcol1.metric("Market Cap Global", f"${fact_global['total_market_cap_usd'].iloc[-1] / 1e12:,.2f}T")
    if "total_volume_usd" in fact_global.columns:
        gcol2.metric("Volumen Global", f"${fact_global['total_volume_usd'].iloc[-1] / 1e9:,.1f}B")
    if "active_cryptocurrencies" in fact_global.columns:
        gcol3.metric("Cryptos Activas", f"{fact_global['active_cryptocurrencies'].iloc[-1]:,.0f}")

st.divider()

# =============================================================
# TOP 10 POR MARKET CAP
# =============================================================
st.subheader("Top 10 por Market Cap")

if "name" in df.columns:
    top10 = (
        df.sort_values("market_cap", ascending=False)
        .drop_duplicates(subset="crypto_id")
        .head(10)
    )

    col_left, col_right = st.columns(2)

    with col_left:
        fig_bar = px.bar(
            top10, x="name", y="market_cap",
            color="current_price", color_continuous_scale="Viridis",
            labels={"market_cap": "Market Cap (USD)", "name": "", "current_price": "Precio"},
        )
        fig_bar.update_layout(height=400, xaxis_tickangle=-45)
        st.plotly_chart(fig_bar, use_container_width=True)

    with col_right:
        fig_pie = px.pie(top10, values="market_cap", names="name", hole=0.4)
        fig_pie.update_layout(height=400)
        st.plotly_chart(fig_pie, use_container_width=True)

st.divider()

# =============================================================
# GANADORES Y PERDEDORES 24H
# =============================================================
if "price_change_percentage_24h" in df.columns and "name" in df.columns:
    st.subheader("Movimiento 24h")

    unique_df = df.drop_duplicates(subset="crypto_id").dropna(subset=["price_change_percentage_24h"])

    col_win, col_lose = st.columns(2)

    with col_win:
        winners = unique_df.nlargest(5, "price_change_percentage_24h")
        fig_w = px.bar(
            winners, x="name", y="price_change_percentage_24h",
            color_discrete_sequence=["#00CC96"],
            labels={"price_change_percentage_24h": "Cambio %", "name": ""},
        )
        fig_w.update_layout(height=300, title="Top 5 Ganadores")
        st.plotly_chart(fig_w, use_container_width=True)

    with col_lose:
        losers = unique_df.nsmallest(5, "price_change_percentage_24h")
        fig_l = px.bar(
            losers, x="name", y="price_change_percentage_24h",
            color_discrete_sequence=["#EF553B"],
            labels={"price_change_percentage_24h": "Cambio %", "name": ""},
        )
        fig_l.update_layout(height=300, title="Top 5 Perdedores")
        st.plotly_chart(fig_l, use_container_width=True)


In [ ]:
%%writefile ../stack/dashboard/pages/2_Gold_Ranking_Precios.py

"""
Ranking y Precios - Tabla interactiva con filtros y comparacion.
Objetivo: responder "cual es el precio de X? como se compara con Y?"
"""

import streamlit as st
import plotly.express as px
from db import load_table, show_last_updated_badge

st.header("Gold - Ranking y Precios")
show_last_updated_badge("gold", "fact_crypto_markets", col="_loaded_at")

# --- Cargar datos ---
fact = load_table("gold", "fact_crypto_markets")
dim = load_table("gold", "dim_crypto")

if fact.empty:
    st.warning("No hay datos en gold todavia. Ejecuta el pipeline desde Airflow.")
    st.stop()

df = fact.copy()
if not dim.empty and "crypto_id" in dim.columns:
    df = df.merge(dim[["crypto_id", "name", "symbol"]], on="crypto_id", how="left")

if "name" not in df.columns:
    st.error("Faltan columnas de nombre. Revisa el pipeline Gold.")
    st.stop()

unique_df = df.drop_duplicates(subset="crypto_id").sort_values("market_cap", ascending=False)

# =============================================================
# FILTROS
# =============================================================
with st.sidebar:
    st.subheader("Filtros")

    # Filtro por rango de precio
    if "current_price" in unique_df.columns:
        price_min = float(unique_df["current_price"].min())
        price_max = float(unique_df["current_price"].max())
        price_range = st.slider(
            "Rango de precio (USD)",
            min_value=price_min, max_value=price_max,
            value=(price_min, price_max), format="$%.2f",
        )
        unique_df = unique_df[
            unique_df["current_price"].between(price_range[0], price_range[1])
        ]

    # Filtro por rank
    if "market_cap_rank" in unique_df.columns:
        max_rank = int(unique_df["market_cap_rank"].max())
        rank_limit = st.slider("Top N por ranking", 5, max(max_rank, 50), 50)
        unique_df = unique_df[unique_df["market_cap_rank"] <= rank_limit]

# =============================================================
# TABLA DE PRECIOS
# =============================================================
st.subheader(f"Tabla de precios ({len(unique_df)} criptos)")

display_cols = ["market_cap_rank", "name", "symbol", "current_price", "market_cap",
                "total_volume", "price_change_percentage_24h", "high_24h", "low_24h"]
available_cols = [c for c in display_cols if c in unique_df.columns]

tabla = unique_df[available_cols].reset_index(drop=True)

col_config = {
    "current_price": st.column_config.NumberColumn("Precio", format="$%.2f"),
    "market_cap": st.column_config.NumberColumn("Market Cap", format="$%.0f"),
    "total_volume": st.column_config.NumberColumn("Volumen 24h", format="$%.0f"),
    "price_change_percentage_24h": st.column_config.NumberColumn("Cambio 24h %", format="%.2f%%"),
    "high_24h": st.column_config.NumberColumn("Max 24h", format="$%.2f"),
    "low_24h": st.column_config.NumberColumn("Min 24h", format="$%.2f"),
    "market_cap_rank": st.column_config.NumberColumn("Rank", format="%d"),
    "name": "Nombre",
    "symbol": "Simbolo",
}

st.dataframe(tabla, column_config=col_config, use_container_width=True, hide_index=True)

st.divider()

# =============================================================
# COMPARADOR
# =============================================================
st.subheader("Comparar criptomonedas")

all_names = sorted(unique_df["name"].dropna().unique())
selected = st.multiselect("Selecciona criptos para comparar", all_names, default=all_names[:3])

if selected:
    compare_df = unique_df[unique_df["name"].isin(selected)]

    metrics = ["current_price", "market_cap", "total_volume"]
    available_metrics = [m for m in metrics if m in compare_df.columns]

    for metric in available_metrics:
        labels = {
            "current_price": "Precio (USD)",
            "market_cap": "Market Cap (USD)",
            "total_volume": "Volumen 24h (USD)",
        }
        fig = px.bar(
            compare_df, x="name", y=metric, color="name",
            labels={metric: labels.get(metric, metric), "name": ""},
        )
        fig.update_layout(height=300, showlegend=False)
        st.plotly_chart(fig, use_container_width=True)


In [ ]:
%%writefile ../stack/dashboard/pages/3_Gold_Volatilidad_Riesgo.py

"""
Volatilidad y Riesgo - Analisis de spread, categorias y dispersion.
Objetivo: responder "cuales son las criptos mas volatiles? donde esta el riesgo?"
"""

import streamlit as st
import plotly.express as px
from db import load_table, show_last_updated_badge

st.header("Gold - Volatilidad y Riesgo")
show_last_updated_badge("gold", "gold_abt_crypto", col="_created_at")

# --- Cargar datos ---
abt = load_table("gold", "gold_abt_crypto")
fact = load_table("gold", "fact_crypto_markets")
dim = load_table("gold", "dim_crypto")

if abt.empty and fact.empty:
    st.warning("No hay datos en gold todavia. Ejecuta el pipeline desde Airflow.")
    st.stop()

# =============================================================
# DISTRIBUCION DE VOLATILIDAD
# =============================================================
if not abt.empty and "volatility_category" in abt.columns:
    st.subheader("Distribucion por volatilidad")

    col_a, col_b = st.columns(2)

    with col_a:
        vol_counts = abt["volatility_category"].value_counts().reset_index()
        vol_counts.columns = ["categoria", "cantidad"]
        fig_vol = px.pie(
            vol_counts, values="cantidad", names="categoria",
            hole=0.4, color="categoria",
            color_discrete_map={"baja": "#00CC96", "media": "#FFA15A", "alta": "#EF553B"},
        )
        fig_vol.update_layout(height=350, title="Volatilidad 24h")
        st.plotly_chart(fig_vol, use_container_width=True)

    with col_b:
        if "market_cap_tier" in abt.columns:
            tier_counts = abt["market_cap_tier"].value_counts().reset_index()
            tier_counts.columns = ["tier", "cantidad"]
            fig_tier = px.pie(
                tier_counts, values="cantidad", names="tier",
                hole=0.4, color="tier",
                color_discrete_map={"top_10": "#636EFA", "top_25": "#AB63FA", "rest": "#B6E880"},
            )
            fig_tier.update_layout(height=350, title="Tiers por Market Cap")
            st.plotly_chart(fig_tier, use_container_width=True)

    st.divider()

# =============================================================
# SPREAD 24H
# =============================================================
df = fact.copy()
if not dim.empty and "crypto_id" in dim.columns:
    df = df.merge(dim[["crypto_id", "name", "symbol"]], on="crypto_id", how="left")

if "spread_pct" in df.columns and "name" in df.columns:
    st.subheader("Spread 24h — Top 15")
    st.caption("Diferencia porcentual entre max y min del dia. Mayor spread = mayor volatilidad intradiaria.")

    spread_data = (
        df[["name", "spread_pct"]]
        .drop_duplicates(subset="name")
        .dropna()
        .sort_values("spread_pct", ascending=False)
        .head(15)
    )

    fig_spread = px.bar(
        spread_data, x="name", y="spread_pct",
        color="spread_pct", color_continuous_scale="YlOrRd",
        labels={"spread_pct": "Spread %", "name": ""},
    )
    fig_spread.update_layout(height=400, xaxis_tickangle=-45)
    st.plotly_chart(fig_spread, use_container_width=True)

    st.divider()

# =============================================================
# SCATTER: PRECIO VS VOLUMEN (bubble = market cap)
# =============================================================
if not abt.empty and "current_price" in abt.columns and "total_volume" in abt.columns:
    st.subheader("Mapa de riesgo: Precio vs Volumen")
    st.caption("Tamano = market cap, Color = volatilidad. Criptos en esquina inferior izq = bajo riesgo.")

    hover = "id" if "id" in abt.columns else None
    color = "volatility_category" if "volatility_category" in abt.columns else None

    fig_scatter = px.scatter(
        abt, x="total_volume", y="current_price",
        size="market_cap", color=color, hover_name=hover,
        log_x=True, log_y=True,
        color_discrete_map={"baja": "#00CC96", "media": "#FFA15A", "alta": "#EF553B"},
        labels={"total_volume": "Volumen 24h (log)", "current_price": "Precio (log)"},
    )
    fig_scatter.update_layout(height=500)
    st.plotly_chart(fig_scatter, use_container_width=True)

    st.divider()

# =============================================================
# DISTANCIA A ATH / ATL
# =============================================================
if not abt.empty and "ath_distance_pct" in abt.columns and "id" in abt.columns:
    st.subheader("Distancia al All-Time High")
    st.caption("Que tan lejos esta cada crypto de su maximo historico. Valores negativos = por debajo del ATH.")

    ath_data = (
        abt[["id", "ath_distance_pct"]]
        .dropna()
        .sort_values("ath_distance_pct", ascending=True)
        .head(20)
    )

    fig_ath = px.bar(
        ath_data, x="id", y="ath_distance_pct",
        color="ath_distance_pct", color_continuous_scale="RdYlGn",
        labels={"ath_distance_pct": "Distancia al ATH (%)", "id": ""},
    )
    fig_ath.update_layout(height=400, xaxis_tickangle=-45)
    st.plotly_chart(fig_ath, use_container_width=True)


In [ ]:
%%writefile ../stack/dashboard/pages/4_Gold_Dominancia.py

"""
Dominancia - Participacion de mercado y concentracion.
Objetivo: responder "quien domina el mercado? esta concentrado o distribuido?"
"""

import streamlit as st
import plotly.express as px
from db import load_table, show_last_updated_badge

st.header("Gold - Dominancia de Mercado")
show_last_updated_badge("gold", "gold_abt_crypto", col="_created_at")

# --- Cargar datos ---
abt = load_table("gold", "gold_abt_crypto")
fact_global = load_table("gold", "fact_global_market")

if abt.empty:
    st.warning("No hay datos ABT en gold todavia. Ejecuta el pipeline desde Airflow.")
    st.stop()

# =============================================================
# BTC / ETH DOMINANCIA
# =============================================================
if not fact_global.empty:
    st.subheader("Dominancia global")

    col1, col2, col3 = st.columns(3)

    btc_dom = fact_global.get("btc_dominance", [None]).iloc[-1] if "btc_dominance" in fact_global.columns else None
    eth_dom = fact_global.get("eth_dominance", [None]).iloc[-1] if "eth_dominance" in fact_global.columns else None

    if btc_dom is not None:
        col1.metric("Bitcoin", f"{btc_dom:.1f}%")
    if eth_dom is not None:
        col2.metric("Ethereum", f"{eth_dom:.1f}%")
    if btc_dom is not None and eth_dom is not None:
        col3.metric("Altcoins", f"{100 - btc_dom - eth_dom:.1f}%")

        # Donut de dominancia
        dom_data = [
            {"Segmento": "Bitcoin", "Dominancia": btc_dom},
            {"Segmento": "Ethereum", "Dominancia": eth_dom},
            {"Segmento": "Altcoins", "Dominancia": 100 - btc_dom - eth_dom},
        ]
        fig_dom = px.pie(
            dom_data, values="Dominancia", names="Segmento", hole=0.5,
            color="Segmento",
            color_discrete_map={"Bitcoin": "#F7931A", "Ethereum": "#627EEA", "Altcoins": "#00CC96"},
        )
        fig_dom.update_layout(height=350)
        st.plotly_chart(fig_dom, use_container_width=True)

    st.divider()

# =============================================================
# MARKET SHARE POR CRIPTO
# =============================================================
if "market_dominance" in abt.columns and "id" in abt.columns:
    st.subheader("Participacion de mercado individual")
    st.caption("Porcentaje del market cap total que representa cada cripto")

    share_df = (
        abt[["id", "market_dominance", "market_cap"]]
        .dropna()
        .sort_values("market_dominance", ascending=False)
    )

    # Top 10 + "Otros"
    top = share_df.head(10).copy()
    rest = share_df.iloc[10:]
    if not rest.empty:
        import pandas as pd
        otros = pd.DataFrame([{
            "id": f"Otros ({len(rest)})",
            "market_dominance": rest["market_dominance"].sum(),
            "market_cap": rest["market_cap"].sum(),
        }])
        top = pd.concat([top, otros], ignore_index=True)

    fig_share = px.bar(
        top, x="id", y="market_dominance",
        color="market_dominance", color_continuous_scale="Blues",
        labels={"market_dominance": "Dominancia %", "id": ""},
    )
    fig_share.update_layout(height=400, xaxis_tickangle=-45)
    st.plotly_chart(fig_share, use_container_width=True)

    st.divider()

# =============================================================
# CONCENTRACION: TOP 10 vs RESTO
# =============================================================
if "market_cap_tier" in abt.columns and "market_cap" in abt.columns:
    st.subheader("Concentracion del mercado")
    st.caption("Que porcentaje del valor total concentran las Top 10, Top 25 y el resto")

    tier_summary = (
        abt.groupby("market_cap_tier")["market_cap"]
        .sum()
        .reset_index()
    )
    tier_summary.columns = ["Tier", "Market Cap"]
    tier_summary["Porcentaje"] = (tier_summary["Market Cap"] / tier_summary["Market Cap"].sum() * 100).round(2)

    col_left, col_right = st.columns(2)

    with col_left:
        fig_conc = px.pie(
            tier_summary, values="Market Cap", names="Tier", hole=0.4,
            color="Tier",
            color_discrete_map={"top_10": "#636EFA", "top_25": "#AB63FA", "rest": "#B6E880"},
        )
        fig_conc.update_layout(height=350)
        st.plotly_chart(fig_conc, use_container_width=True)

    with col_right:
        for _, row in tier_summary.iterrows():
            st.metric(row["Tier"], f"{row['Porcentaje']:.1f}%", f"${row['Market Cap'] / 1e9:,.1f}B")

# =============================================================
# PRICE TIERS
# =============================================================
if "price_tier" in abt.columns:
    st.divider()
    st.subheader("Distribucion por rango de precio")
    st.caption("micro (<$1), small ($1-100), medium ($100-10K), large (>$10K)")

    tier_data = abt["price_tier"].value_counts().reset_index()
    tier_data.columns = ["Tier", "Cantidad"]

    fig_pt = px.bar(
        tier_data, x="Tier", y="Cantidad", color="Tier",
        color_discrete_map={"micro": "#B6E880", "small": "#FFA15A", "medium": "#636EFA", "large": "#EF553B"},
    )
    fig_pt.update_layout(height=300, showlegend=False)
    st.plotly_chart(fig_pt, use_container_width=True)


---

## 🎉 ¡Capa Gold completada!

Acabás de construir un Star Schema (BI-friendly) y una ABT (ML-friendly), y los visualizaste en un dashboard. Esto **cierra el pipeline Medallion** — Bronze → Silver → Gold → Consumo.

**Próximo paso (con entrega)**: practicá los fundamentos de SQL de Gold (agregaciones, JOIN *star*, CASE, ranking) sobre Northwind en el [Ejercicio de la Clase 05](ejercicios/ejercicio.ipynb).
